# Assignment: SQL Notebook for Peer Assignment


## Introduction
Using this Python notebook you will:

1.  Understand the Spacex DataSet
2.  Load the dataset  into the corresponding table in a Db2 database
3.  Execute SQL queries to answer assignment questions 

## Overview of the DataSet

SpaceX has gained worldwide attention for a series of historic milestones. 

It is the only private company ever to return a spacecraft from low-earth orbit, which it first accomplished in December 2010.
SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars where as other providers cost upward of 165 million dollars each, much of the savings is because Space X can reuse the first stage. 


Therefore if we can determine if the first stage will land, we can determine the cost of a launch. 

This information can be used if an alternate company wants to bid against SpaceX for a rocket launch.

This dataset includes a record for each payload carried during a SpaceX mission into outer space.

### Download the datasets

This assignment requires you to load the spacex dataset.

In many cases the dataset to be analyzed is available as a .CSV (comma separated values) file, perhaps on the internet. Click on the link below to download and save the dataset (.CSV file):

 <a href="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv" target="_blank">Spacex DataSet</a>

### Connect to the database

Let us first load the SQL extension and establish a connection with the database

In [ ]:
%pip install sqlalchemy ipython-sql prettytable pandas

In [1]:
%load_ext sql

In [2]:
import csv, sqlite3
import prettytable
import pandas as pd
prettytable.DEFAULT = "DEFAULT"

con = sqlite3.connect("datasets/my_data1.db")
cun = con.cursor()

In [3]:
%sql sqlite:///datasets/my_data1.db

In [4]:
df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv")
df.to_sql("SPACEXTBL", con, if_exists='replace', index=False,method="multi")

101

In [5]:
# Drop the table if exists
%sql DROP TABLE IF EXISTS SPACEXTABLE;
%sql create table SPACEXTABLE as select * from SPACEXTBL where Date is not null

 * sqlite:///datasets/my_data1.db
Done.
 * sqlite:///datasets/my_data1.db
Done.


[]

## Tasks

Now write and execute SQL queries to solve the assignment tasks.

**Note: If the column names are in mixed case enclose it in double quotes
   For Example "Landing_Outcome"**

### Task 1
##### Display the names of the unique launch sites  in the space mission

In [ ]:
%sql select distinct Launch_Site from SPACEXTABLE

 * sqlite:///datasets/my_data1.db
Done.


Launch_Site
CCAFS LC-40
VAFB SLC-4E
KSC LC-39A
CCAFS SLC-40


### Task 2
#####  Display 5 records where launch sites begin with the string 'CCA'

In [12]:
%sql select * from SPACEXTABLE where Launch_Site like '%CCA%' limit 5

 * sqlite:///datasets/my_data1.db
Done.


Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of Brouere cheese",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


### Task 3

##### Display the total payload mass carried by boosters launched by NASA (CRS)


In [21]:
%sql select sum("PAYLOAD_MASS__KG_") as total from SPACEXTABLE where Customer = 'NASA (CRS)'

 * sqlite:///datasets/my_data1.db
Done.


total
45596


### Task 4

##### Display average payload mass carried by booster version F9 v1.1

In [24]:
%sql select avg("PAYLOAD_MASS__KG_") as average from SPACEXTABLE where Booster_Version = 'F9 v1.1'


 * sqlite:///datasets/my_data1.db
Done.


average
2928.4


### Task 5

##### List the date when the first succesful landing outcome in ground pad was acheived.

_Hint:Use min function_ 

In [30]:
%sql select min(Date) from SPACEXTABLE where Mission_Outcome = 'Success' and Landing_Outcome = 'Success (ground pad)'

 * sqlite:///datasets/my_data1.db
Done.


min(Date)
2015-12-22


### Task 6

##### List the names of the boosters which have success in drone ship and have payload mass greater than 4000 but less than 6000

In [37]:
%%sql 
select distinct Booster_Version from SPACEXTABLE 
where Mission_Outcome = 'Success' and Landing_Outcome = 'Success (drone ship)'
and PAYLOAD_MASS__KG_ between 4000 and 6000

 * sqlite:///datasets/my_data1.db
Done.


Booster_Version
F9 FT B1022
F9 FT B1026
F9 FT B1021.2
F9 FT B1031.2


### Task 7

##### List the total number of successful and failure mission outcomes

In [38]:
%%sql
select Mission_Outcome, count(*) from SPACEXTABLE
group by Mission_Outcome

 * sqlite:///datasets/my_data1.db
Done.


Mission_Outcome,count(*)
Failure (in flight),1
Success,98
Success,1
Success (payload status unclear),1


### Task 8

##### List all the booster_versions that have carried the maximum payload mass. Use a subquery.

In [41]:
%%sql
select distinct Booster_Version from SPACEXTABLE
where PAYLOAD_MASS__KG_ = (
    select max(PAYLOAD_MASS__KG_) from SPACEXTABLE
)

 * sqlite:///datasets/my_data1.db
Done.


Booster_Version
F9 B5 B1048.4
F9 B5 B1049.4
F9 B5 B1051.3
F9 B5 B1056.4
F9 B5 B1048.5
F9 B5 B1051.4
F9 B5 B1049.5
F9 B5 B1060.2
F9 B5 B1058.3
F9 B5 B1051.6


### Task 9


##### List the records which will display the month names, failure landing_outcomes in drone ship ,booster versions, launch_site for the months in year 2015.

**Note: SQLLite does not support monthnames. So you need to use  substr(Date, 6,2) as month to get the months and substr(Date,0,5)='2015' for year.**

In [45]:
%%sql
select substr(Date, 6,2) as month, Landing_Outcome, Booster_Version, Launch_Site,  * from SPACEXTABLE
where substr(Date,0,5)='2015' and Landing_Outcome = 'Failure (drone ship)'

 * sqlite:///datasets/my_data1.db
Done.


month,Landing_Outcome,Booster_Version,Launch_Site,Date,Time (UTC),Booster_Version_1,Launch_Site_1,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome_1
01,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40,2015-01-10,9:47:00,F9 v1.1 B1012,CCAFS LC-40,SpaceX CRS-5,2395,LEO (ISS),NASA (CRS),Success,Failure (drone ship)
04,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40,2015-04-14,20:10:00,F9 v1.1 B1015,CCAFS LC-40,SpaceX CRS-6,1898,LEO (ISS),NASA (CRS),Success,Failure (drone ship)


### Task 10

##### Rank the count of landing outcomes (such as Failure (drone ship) or Success (ground pad)) between the date 2010-06-04 and 2017-03-20, in descending order.

In [48]:
%%sql
select Landing_Outcome, count(*) from SPACEXTABLE
where date between "2010-06-04" and "2017-03-20"
group by Landing_Outcome
order by count(*) desc

 * sqlite:///datasets/my_data1.db
Done.


Landing_Outcome,count(*)
No attempt,10
Success (drone ship),5
Failure (drone ship),5
Success (ground pad),3
Controlled (ocean),3
Uncontrolled (ocean),2
Failure (parachute),2
Precluded (drone ship),1
